# Demo — Onde o vetor mora: Azure Database for PostgreSQL (pgvector)

**Objetivo:** guardar os trechos do acervo, com os seus vetores, numa tabela comum do PostgreSQL e buscar por similaridade com o pgvector. É a linha **PostgreSQL (pgvector)** do slide 22 e a primeira resposta do slide 21: o vetor vive junto do dado operacional, no banco que já existe. O índice é o HNSW do slide 23, e o slide 20 dá o tom: a pergunta não é qual banco vetorial é o melhor, é se o dado muda em tempo real.

1. **Preparar os vetores** → os mesmos trechos e o mesmo embedding das outras bases da série;
2. **Criar a estrutura vetorial** → uma tabela com uma coluna do tipo `vector` e um índice HNSW;
3. **Gravar os trechos** → `INSERT ... ON CONFLICT` pelo `id` do trecho, numa transação;
4. **Buscar por similaridade** → `ORDER BY embedding <=> pergunta`, com duas perguntas do gabarito;
5. **RAG completo** → a resposta com citação a partir dos 3 primeiros trechos.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** um servidor flexível do **Azure Database for PostgreSQL** (as versões 13 a 18 trazem o pgvector 0.8.2; o tier Burstable basta para a aula), com a extensão **`vector` liberada no parâmetro de servidor `azure.extensions`** e o IP desta máquina nas regras de firewall do servidor.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 5)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`
- `AZURE_POSTGRES_CONNECTION_STRING`, no formato do libpq: `host=<servidor>.postgres.database.azure.com port=5432 dbname=postgres user=<usuario> password=<senha> sslmode=require`

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução. Use o usuário administrador do servidor: ele é membro do papel `azure_pg_admin`, que pode criar a extensão.

> Não grave chaves diretamente no notebook. A string de conexão também carrega a senha.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "numpy>=1.26" "psycopg[binary]>=3.2" "pgvector>=0.5"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
import numpy as np
import psycopg
from pgvector.psycopg import register_vector

AZURE_POSTGRES_CONNECTION_STRING = os.getenv("AZURE_POSTGRES_CONNECTION_STRING") or getpass(
    "AZURE_POSTGRES_CONNECTION_STRING (host=... port=5432 dbname=... user=... password=... sslmode=require): "
)

# autocommit: cada comando vale na hora.
# Se o IP desta máquina não estiver no firewall do servidor, a recusa é imediata: "no pg_hba.conf entry for host ...".
# Se a rede bloquear a saída pela porta 5432, a conexão desiste em 15 segundos.
conexao = psycopg.connect(AZURE_POSTGRES_CONNECTION_STRING, autocommit=True, connect_timeout=15)

# A extensão se chama "vector" (pgvector é o nome do projeto) e precisa estar liberada em azure.extensions.
try:
    conexao.execute("CREATE EXTENSION IF NOT EXISTS vector")
except psycopg.Error as erro:
    raise RuntimeError(
        "Não foi possível criar a extensão vector. Libere 'vector' no parâmetro de servidor "
        "azure.extensions e rode esta célula de novo."
    ) from erro

# Só depois da extensão: o psycopg passa a enviar arrays do numpy como o tipo vector (uma lista iria como double precision[]).
register_vector(conexao)

versao_pgvector = conexao.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector'").fetchone()[0]
print("PostgreSQL:", conexao.info.host, "| banco:", conexao.info.dbname)
print("Versão do servidor:", conexao.execute("SHOW server_version").fetchone()[0], "| pgvector:", versao_pgvector)

### Funções auxiliares

As mesmas dos outros notebooks do Cap. 5 que as usam, sem mudança:

- `carregar_trechos`: um trecho por seção `## ` de cada documento do acervo;
- `texto_para_embedding` e `gerar_embeddings`: o texto que vira vetor e a chamada ao deployment de embedding, com a dimensão do contrato;
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu;
- `responder`: a geração com citação, usada na Execução 5.

In [ ]:
import json
import re

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


def texto_para_embedding(trecho: dict) -> str:
    """O texto que vira vetor: título do documento, seção e conteúdo."""
    return f"{trecho['titulo']} — {trecho['secao']}\n{trecho['conteudo']}"


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

---

# Execução 1 — Preparar os vetores

Os mesmos 50 trechos e o mesmo embedding das outras bases da série e do `tipos-de-rag/00`: um trecho por seção `## `, vetorizado com título, seção e conteúdo. O vetor não depende da base. O que muda daqui em diante é só **onde ele mora**.

In [ ]:
trechos = carregar_trechos(PASTA_ACERVO)
vetores = gerar_embeddings([texto_para_embedding(t) for t in trechos])

print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")
print(f"{len(vetores)} vetores, cada um com {len(vetores[0])} dimensões")

---

# Execução 2 — Criar a estrutura vetorial

No PostgreSQL, a estrutura vetorial é uma **tabela comum**, `cap05_trechos`:

- colunas relacionais: `id` (chave primária), `doc_id`, `titulo`, `secao`, `conteudo` e `fonte`;
- a coluna `embedding`, do tipo `vector(EMBEDDING_DIMENSIONS)`: a dimensão faz parte do tipo, e o banco recusa vetor de outro tamanho;
- um índice **HNSW** com `vector_cosine_ops`: a classe de operadores precisa casar com o operador da consulta (`<=>`, a distância de cosseno), senão o banco nem considera o índice.

A dimensão entra no `CREATE TABLE` como literal inteiro, montada pelo `psycopg.sql`: SQL não se monta concatenando texto.

In [ ]:
from psycopg import sql

conexao.execute(sql.SQL("""
    CREATE TABLE IF NOT EXISTS cap05_trechos (
        id        text PRIMARY KEY,
        doc_id    text NOT NULL,
        titulo    text NOT NULL,
        secao     text NOT NULL,
        conteudo  text NOT NULL,
        fonte     text NOT NULL,
        embedding vector({dimensoes}) NOT NULL
    )
""").format(dimensoes=sql.Literal(EMBEDDING_DIMENSIONS)))

conexao.execute("""
    CREATE INDEX IF NOT EXISTS cap05_trechos_embedding_hnsw
    ON cap05_trechos USING hnsw (embedding vector_cosine_ops)
""")

print("Tabela 'cap05_trechos', como o banco a registrou:")
display(pd.DataFrame(
    conexao.execute("""
        SELECT attname, format_type(atttypid, atttypmod)
        FROM pg_attribute
        WHERE attrelid = 'cap05_trechos'::regclass AND attnum > 0 AND NOT attisdropped
        ORDER BY attnum
    """).fetchall(),
    columns=["coluna", "tipo"],
))
display(pd.DataFrame(
    conexao.execute("SELECT indexname, indexdef FROM pg_indexes WHERE tablename = 'cap05_trechos' ORDER BY indexname").fetchall(),
    columns=["índice", "definição"],
))

### O que observar

- O tipo registrado é `vector(1536)`, ou a dimensão do seu `.env`: a dimensão do contrato agora está escrita no esquema da tabela; o modelo, não.
- A tabela tem dois índices: o B-tree da chave primária, que todo banco relacional tem, e o HNSW do vetor. **O vetor é só mais uma coluna.**

### Os limites desta base (slide 22)

| Serviço | Dimensões | Índices | Reranking | Quando é a escolha certa |
|---|---|---|---|---|
| PostgreSQL (pgvector) | até 16.000¹ | HNSW, IVFFlat, DiskANN | Não | Casa relacional, alta dimensão, escala horizontal |

¹ O pgvector armazena até 16.000 dimensões, mas o índice HNSW aceita até 2.000 (4.000 com halfvec).

Os 1.536 números do `text-embedding-3-small` cabem no HNSW. Os 3.072 do `text-embedding-3-large`, não: o `CREATE INDEX` falha, e as saídas são `halfvec`, DiskANN com quantização (em preview) ou reduzir a dimensão.

### O algoritmo do índice (slide 23)

| | HNSW | IVFFlat | DiskANN |
|---|---|---|---|
| Como funciona | Grafo em camadas, em memória | Particiona o espaço em listas | Grafo otimizado para SSD |
| Recall | Alto | Médio | Alto |
| Memória | Alta | Baixa | Baixa |
| Dado que muda | Lida bem | Pede reconstrução | Lida bem |
| No PostgreSQL do Azure | `USING hnsw`, do pgvector; pode ser criado com a tabela vazia | `USING ivfflat ... WITH (lists = ...)`; as listas saem de um k-means sobre os dados, então o índice vem depois da carga | Extensão `pg_diskann` (PostgreSQL 14 ou superior), também liberada em `azure.extensions`; `USING diskann` |

Sem índice, o pgvector faz busca **exata**: recall perfeito, ao custo de comparar a pergunta com todas as linhas. O índice troca um pouco de recall por velocidade.

---

# Execução 3 — Gravar os trechos

Cada trecho entra com o seu vetor por um `INSERT ... ON CONFLICT (id) DO UPDATE`: como o `id` do trecho é estável (documento + número da seção), rodar a célula de novo **atualiza** as mesmas linhas, em vez de duplicar.

A carga roda numa transação só: ou os 50 trechos entram, ou nenhum entra. É o mesmo comando que gravaria um pedido ou um cadastro, e é isso que o slide 21 chama de "o vetor vive junto do dado operacional": o vetor e o dado mudam juntos, sem um índice separado para sincronizar.

**O que esperar:** 50 linhas na tabela, também na segunda execução. Diferente do AI Search, a contagem já sai certa: o que foi confirmado (`COMMIT`) aparece na consulta seguinte.

In [ ]:
linhas = [
    (t["id"], t["doc_id"], t["titulo"], t["secao"], t["conteudo"], t["fonte"], np.array(v, dtype=np.float32))
    for t, v in zip(trechos, vetores)
]

with conexao.transaction(), conexao.cursor() as cursor:
    cursor.executemany("""
        INSERT INTO cap05_trechos (id, doc_id, titulo, secao, conteudo, fonte, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (id) DO UPDATE SET
            doc_id = EXCLUDED.doc_id,
            titulo = EXCLUDED.titulo,
            secao = EXCLUDED.secao,
            conteudo = EXCLUDED.conteudo,
            fonte = EXCLUDED.fonte,
            embedding = EXCLUDED.embedding
    """, linhas)

total = conexao.execute("SELECT count(*) FROM cap05_trechos").fetchone()[0]
print(f"Trechos enviados: {len(linhas)} | linhas na tabela: {total} (trechos no acervo: {len(trechos)})")
if total > len(trechos):
    print("Há linhas a mais: sobras de uma carga anterior. O ON CONFLICT atualiza, mas não apaga nada.")

---

# Execução 4 — Buscar por similaridade

A pergunta vira vetor com o mesmo `gerar_embeddings`, e o SQL ordena os trechos pela distância até ela: `ORDER BY embedding <=> pergunta LIMIT k`.

- `<=>` é a **distância de cosseno**: 0 quando os vetores apontam na mesma direção, até 2 quando apontam em direções opostas.
- A **similaridade** é `1 - distância`, o próprio cosseno: quanto maior, mais perto. O AI Search mostra outra escala (`1 / (1 + distância)`), mas a ordem é a mesma.

Rodamos duas perguntas do gabarito: uma em linguagem natural, em que o vetor tende a acertar, e a do SKU 48213-B, uma string exata, do lado em que o vetor erra no slide 14.

### O que esperar e por que pode variar

- **Trocar a base não muda a natureza do vetor.** Com o mesmo modelo e a mesma dimensão, a ordem tende a ser a dos outros notebooks da série. O quase empate do slide 14 continua aqui: 48213-B, C e D aparecem juntos, com similaridades muito próximas, e o certo pode vir em 1º ou não.
- Os números mudam com o modelo e a dimensão do embedding. E é o PostgreSQL que decide se usa o índice HNSW (busca aproximada) ou lê a tabela inteira (busca exata): a última linha da célula mostra o caminho que ele escolheu. Com só 50 linhas, a busca tende a ser exata, e então o que varia de uma execução para outra é o embedding, não o índice.

In [ ]:
from psycopg.rows import dict_row


def buscar(pergunta: str, k: int = 5) -> list[dict]:
    """Os k trechos mais próximos da pergunta, pela distância de cosseno (<=>) do pgvector."""
    vetor = np.array(gerar_embeddings([pergunta])[0], dtype=np.float32)
    with conexao.cursor(row_factory=dict_row) as cursor:
        return cursor.execute("""
            SELECT id, titulo, secao, conteudo, fonte, 1 - (embedding <=> %(vetor)s) AS similaridade
            FROM cap05_trechos
            ORDER BY embedding <=> %(vetor)s
            LIMIT %(k)s
        """, {"vetor": vetor, "k": k}).fetchall()


resultados = {pid: buscar(PERGUNTAS[pid]["pergunta"], k=5) for pid in ["linguagem-natural", "string-exata"]}

for pid, top5 in resultados.items():
    esperados = PERGUNTAS[pid]["trechos_esperados"][0]
    lugar = posicao([t["id"] for t in top5], esperados)
    situacao = f"{lugar}º lugar" if lugar else "fora do top 5"
    display(Markdown(f"**{PERGUNTAS[pid]['pergunta']}** ({PERGUNTAS[pid]['categoria']}) → trecho certo: {situacao}"))
    display(pd.DataFrame([
        {
            "posição": i,
            "similaridade": round(t["similaridade"], 4),
            "diferença para o 1º": round(t["similaridade"] - top5[0]["similaridade"], 4),
            "trecho": f"{t['titulo']} — {t['secao']}",
            "é o trecho certo?": "sim" if t["id"] in esperados else "",
        }
        for i, t in enumerate(top5, start=1)
    ]))

# O caminho que o PostgreSQL escolheu: o índice HNSW (aproximado) ou a leitura da tabela inteira (exato).
vetor = np.array(gerar_embeddings([PERGUNTAS["string-exata"]["pergunta"]])[0], dtype=np.float32)
plano = conexao.execute("EXPLAIN (COSTS OFF) SELECT id FROM cap05_trechos ORDER BY embedding <=> %s LIMIT 5", (vetor,))
print("Plano de execução:", next(linha for (linha,) in plano if "Scan" in linha).replace("->", "").strip())

### O que observar

- Na pergunta em linguagem natural, o trecho "Como pedir o reembolso de uma viagem" tende a vir no topo.
- Na do SKU 48213-B, veja a coluna **diferença para o 1º**: os SKUs irmãos ficam colados. O vetor captura o assunto (prazo de entrega de rolamento), não o identificador, e isso não muda com a base.
- **Híbrido e reranking são decisões de recuperação**, as dos notebooks de `tipos-de-rag`. **A base é decisão de operação** (slide 21): nenhum banco conserta no vetor o que só a palavra-chave resolve.
- No plano, `Seq Scan on cap05_trechos` quer dizer que o banco comparou a pergunta com todas as linhas: busca exata. Com só 50 linhas, tende a ser esse o caminho, porque comparar com todas sai mais barato que percorrer o grafo. `Index Scan using cap05_trechos_embedding_hnsw` seria a busca aproximada pelo HNSW, que passa a compensar quando a tabela cresce.

---

# Execução 5 — RAG completo

Os 3 primeiros trechos da pergunta em linguagem natural vão para o modelo, que responde citando o `id` de cada trecho usado. A geração é a mesma das outras bases da série e de `tipos-de-rag`: a base só muda de onde os trechos vieram.

In [ ]:
pergunta = PERGUNTAS["linguagem-natural"]["pergunta"]
top3 = resultados["linguagem-natural"][:3]
lugar = posicao([t["id"] for t in top3], PERGUNTAS["linguagem-natural"]["trechos_esperados"][0])
enviados = ", ".join(f"`{t['id']}`" for t in top3)
situacao = f"sim, em {lugar}º lugar" if lugar else "não"

display(Markdown(f"### {pergunta}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta, top3)))

### O que observar

- Leia a linha **O trecho certo está entre eles?** antes de julgar a resposta: ela separa erro de recuperação de erro de geração (slide 29).
- A geração (`responder`, o prompt e o modelo) é a mesma das outras bases da série. Se o trecho certo ficou de fora, o erro é de recuperação, e outra base não o corrige: o vetor é o mesmo.

### Leitura arquitetural

- Slide 22, última coluna, para o PostgreSQL: **"Casa relacional, alta dimensão, escala horizontal"**.
  - *Casa relacional:* o vetor virou mais uma coluna. Dá para filtrar com `WHERE`, juntar com `JOIN` e gravar na mesma transação do pedido ou do cadastro.
  - *Alta dimensão:* o tipo `vector` guarda até 16.000 dimensões; quem tem limite é o índice (2.000 no HNSW).
  - *Escala horizontal:* com os elastic clusters (o Citus gerenciado), os dados se distribuem por vários nós.
- Slide 21, **o dado muda em tempo real?** Se muda, o vetor vive junto do dado operacional, no banco que já existe: o que a Execução 3 gravou apareceu na Execução 4 sem nenhum índice separado para sincronizar.
- **Quem opera, monitora e paga:** o time que já sustenta o PostgreSQL, o mesmo que dimensiona o servidor, acompanha CPU, memória e disco e cuida de backup e de atualização de versão. Não há conta nova: o vetor e o índice HNSW ocupam disco e memória do mesmo servidor, e é ele que cresce quando o acervo cresce.
- **O híbrido não vem pronto:** o PostgreSQL tem busca textual própria, mas juntar texto e vetor (por exemplo, com RRF, como o AI Search faz) fica a cargo do seu SQL ou da aplicação. Reranker embutido também não há (slide 22, coluna Reranking).

### Fala sugerida

> “A pergunta não é qual banco vetorial é o melhor. É se o dado muda em tempo real.”

---

# Quadro final — o que muda ao trocar a base?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Os mesmos trechos e os mesmos vetores dos outros notebooks | O vetor não depende da base: modelo e dimensão continuam sendo o contrato |
| 2 | Uma tabela comum, com a coluna `vector` ao lado das colunas relacionais e um índice HNSW | No PostgreSQL, o vetor é mais uma coluna: mesmo banco, mesmo backup, mesma segurança |
| 3 | `INSERT ... ON CONFLICT` grava e atualiza pelo `id`, numa transação, e a contagem já sai certa | O vetor muda junto com o dado, sem índice separado para sincronizar (slide 21) |
| 4 | `<=>` devolve a distância de cosseno, e o quase empate do slide 14 continua lá | Trocar a base não conserta a recuperação: híbrido e reranking são decisões dos tipos de RAG |
| 5 | A resposta com citação não depende de onde o vetor mora | Escolher pela operação: quem já opera PostgreSQL ganha busca vetorial sem componente novo |

## Frase de fechamento

> **Escolher a base é decisão de operação, não de benchmark: cada componente novo precisa de alguém para operar, monitorar e pagar.**

---

# Referências oficiais

- Azure Database for PostgreSQL — busca vetorial com pgvector:  
  https://learn.microsoft.com/azure/postgresql/extensions/how-to-use-pgvector

- Azure Database for PostgreSQL — liberar extensões no parâmetro azure.extensions:  
  https://learn.microsoft.com/azure/postgresql/extensions/how-to-allow-extensions

- Azure Database for PostgreSQL — desempenho do pgvector: HNSW, IVFFlat, DiskANN e o limite de 2.000 dimensões:  
  https://learn.microsoft.com/azure/postgresql/extensions/how-to-optimize-performance-pgvector

- Azure Architecture Center — como escolher o serviço do Azure para busca vetorial:  
  https://learn.microsoft.com/azure/architecture/guide/technology-choices/vector-search

---

# Limpeza (opcional)

A célula abaixo apaga só o que este notebook criou: a tabela `cap05_trechos`, e o índice HNSW vai junto. A extensão `vector` fica, porque outras tabelas do banco podem usá-la. O servidor também fica, e ele cobra enquanto existir: pare ou apague o servidor no portal quando não for mais usá-lo.

Mudou a dimensão do embedding no `.env`? Reinicie o kernel, porque os notebooks leem o `.env` só uma vez. Com a dimensão nova, a tabela continua com a dimensão anterior, e a carga falha com `expected 1536 dimensions, not ...`: apague a tabela aqui e rode o notebook de novo, com `APAGAR_RECURSOS` de volta em `False`.

Mudou só o modelo, com a mesma dimensão? Aí nada falha: `vector(n)` guarda a dimensão, não o modelo, e o banco não percebe a troca. Reinicie o kernel, para o `.env` novo valer, e rode o notebook inteiro: a Execução 3 regrava os 50 vetores com o modelo novo. É o "reindexar tudo" do slide 10: vetores de modelos diferentes não se comparam.

In [ ]:
APAGAR_RECURSOS = False  # True apaga a tabela cap05_trechos (e o índice HNSW junto)

if APAGAR_RECURSOS:
    conexao.execute("DROP TABLE IF EXISTS cap05_trechos")
    print("Tabela 'cap05_trechos' apagada. A extensão vector e o servidor continuam.")
else:
    print("Nada foi apagado. A tabela 'cap05_trechos' continua no banco.")

conexao.close()
print("Conexão com o PostgreSQL encerrada.")